# Within-Domain Scenario
## Description

For the within-domain scenario, we developed two complementary recommender systems separately for books and movies. First, we separated the ratings data for collaborative filtering (CF) into train and test sets to perform cross-validation at the end. Then, we trained CF models using ALS on user-item rating data within each domain to capture patterns of user preferences based on past interactions, while avoiding items the user had already interacted with. Then, we built content-based filtering (CBF) pipelines by training LDA to generate topic distributions for each item’s tags or descriptions, followed by K-Means clustering to group similar items. Within each cluster, we recommended items to users based on proximity in content space. 

After training collaborative filtering (CF) and content-based filtering (CBF) models within each domain, we combined them into a single hybrid recommendation system. Specifically, we boosted CF scores for items that appeared in a user’s CBF recommendations, giving an extra weight to items that aligned with both collaborative and content signals. This score boosting was capped at 1 to maintain normalized scores. If a user had fewer than the desired number of recommendations (15), we filled in the shortfall using fallback recommendations drawn purely from their CBF hits, ensuring every user received a complete recommendation list. This combination leveraged the strengths of both CF (capturing collective user behavior) and CBF (capturing item similarity), while the fallback guaranteed robustness by covering cases where CF alone could not provide enough high-quality items. By combining CF and CBF within both domains, we aimed to leverage both collaborative and content signals to produce more robust and diverse recommendations tailored to books and movies individually.

## Set up

In [1]:
# Import the findspark module 
import findspark

# Initialize via the full spark path
findspark.init("/usr/local/spark/")
#findspark.init("C:/spark/")

In [2]:
# Import the SparkSession module
from pyspark.sql import SparkSession

# Import the collections module
import collections

# Gets an existing :class:`SparkSession` or, if there is no existing one, creates a
# new one based on the options set in this builder.
spark = SparkSession.builder \
   .master("local[8]") \
   .appName("Project-within-domain") \
   .getOrCreate()

# Main entry point for Spark functionality. A SparkContext represents the
# connection to a Spark cluster, and can be used to create :class:`RDD` and
# broadcast variables on that cluster.
sc = spark.sparkContext

In [3]:
# Imports for content-based filtering
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.ml.feature import CountVectorizer
from pyspark.ml.clustering import LDA, KMeans

In [4]:
# Imports for collaborative filtering
from pyspark.sql.functions import explode, col, lit, array, when, collect_list, collect_set, size, min as spark_min, max as spark_max, array_contains
from pyspark.sql.types import ArrayType, LongType
from pyspark.ml.recommendation import ALS
from pyspark.sql.functions import udf

In [5]:
# Imports for combining 2 models
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.ml.linalg import Vectors
from pyspark.ml.linalg import DenseVector

# Books
## Collaborative Filtering

#### Short Summary for CF
1. Loading of the data  
2. Splitting the data into train and test sets  
3. Model Training (ALS)  
4. Excluding books the user already rated  
5. Normalizing the scores to be between 0 and 1  

### 1. Loading of the data

In [6]:
# Loading book_ratings_old.json
try: 
    book_ratings = spark.read.json("./within_domain_data/books_ratings_old.json")
except Exception as e:
    print(f"Error loading file: {e}")
    raise

book_ratings.show(5)

+--------+------+-------+
| item_id|rating|user_id|
+--------+------+-------+
|41335427|     5|      0|
|41335427|     3|      1|
|41335427|     5|      2|
|41335427|     5|      3|
|41335427|     5|      4|
+--------+------+-------+
only showing top 5 rows



In [8]:
ratings_df = book_ratings.select(
    col("item_id").cast("integer"),
    col("rating").cast("float"),
    col("user_id").cast("integer"),
)

ratings_df.show(10)

+--------+------+-------+
| item_id|rating|user_id|
+--------+------+-------+
|41335427|   5.0|      0|
|41335427|   3.0|      1|
|41335427|   5.0|      2|
|41335427|   5.0|      3|
|41335427|   5.0|      4|
|41335427|   4.0|      5|
|41335427|   5.0|      6|
|41335427|   5.0|      7|
|41335427|   5.0|      8|
|41335427|   5.0|      9|
+--------+------+-------+
only showing top 10 rows



## 2. Splitting the data into train and test sets

In [9]:
# 1. Add binary label for positive (rating >=4) or negative
ratings_labeled = ratings_df.withColumn(
    "label", F.when(F.col("rating") >= 4, 1).otherwise(0)
)

# 2. Add a random number per rating (seeded for reproducibility)
ratings_labeled = ratings_labeled.withColumn(
    "rand", F.rand(seed=42)
)

# 3. For positives: split 80% train, 20% test
pos_train = ratings_labeled.filter((F.col("label") == 1) & (F.col("rand") <= 0.8))
pos_test  = ratings_labeled.filter((F.col("label") == 1) & (F.col("rand") > 0.8))

# 4. For negatives: same split
neg_train = ratings_labeled.filter((F.col("label") == 0) & (F.col("rand") <= 0.8))
neg_test  = ratings_labeled.filter((F.col("label") == 0) & (F.col("rand") > 0.8))

# 5. Combine train and test splits separately
train_df = pos_train.union(neg_train).drop("label", "rand")
test_df = pos_test.union(neg_test).drop("label", "rand")

train_df.show(5)

+--------+------+-------+
| item_id|rating|user_id|
+--------+------+-------+
|41335427|   5.0|      0|
|41335427|   5.0|      3|
|41335427|   5.0|      4|
|41335427|   4.0|      5|
|41335427|   5.0|      7|
+--------+------+-------+
only showing top 5 rows



## 3. Model Training (ALS) - Collaborative Filtering

In [9]:
# ALS works with ratings DataFrame with columns: user, item, rating
als = ALS(
    userCol="user_id", 
    itemCol="item_id", 
    ratingCol="rating",
    rank=30,         # latent factors dimensionality
    maxIter=10, 
    regParam=0.1, 
    coldStartStrategy="drop",  # avoids NaNs in predictions
    implicitPrefs=False,       # we have explicit ratings
    nonnegative=True
)

als_model = als.fit(train_df)

In [10]:
user_recs = als_model.recommendForAllUsers(30)  # top 30 recommendations per user
user_recs.show(2, truncate=False)

+-------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|user_id|recommendations                                                                                                                                                                                                                                                                                                          

### 4. Excluding books the user already rated

In [11]:
# 1. Build mapping user -> set of items they already rated
user_seen_items = train_df.groupBy("user_id").agg(
    collect_set("item_id").alias("seen_items")
)

# 2. Explode user recommendations (array of structs -> individual rows)
recs_exploded = user_recs.withColumn(
    "rec", explode(col("recommendations"))
).select(
    "user_id",
    col("rec.item_id").alias("item_id"),
    col("rec.rating").cast("double").alias("score")
)

# 3. Join exploded recommendations with seen items per user
recs_with_seen = recs_exploded.join(
    user_seen_items, on="user_id", how="left"
)

# 4. Filter out recommendations the user has already rated
recs_filtered = recs_with_seen.filter(
    ~array_contains(col("seen_items"), col("item_id"))
)

# 5. Reconstruct recommendations per user: list of [item_id, score]
user_recs_filtered = recs_filtered.groupBy("user_id").agg(
    collect_list(array("item_id", "score")).alias("recommendations_filtered")
)

# 6. Show example results
user_recs_filtered.show(5, truncate=False)

+-------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|

### 5. Normalizing the scores (0-1)

In [12]:
# 1. Compute global min and max predicted scores across all recommendations
global_minmax = recs_filtered.agg(
    spark_min("score").alias("global_min"),
    spark_max("score").alias("global_max")
).collect()[0]

global_min = global_minmax["global_min"]
global_max = global_minmax["global_max"]

print(f"Global prediction min: {global_min}, max: {global_max}")

# 2. Broadcast min/max for distributed normalization
global_min_b = spark.sparkContext.broadcast(global_min)
global_max_b = spark.sparkContext.broadcast(global_max)

# 3. Normalize scores with min-max scaling
recs_normalized = recs_filtered.withColumn(
    "normalized_score",
    when(
        lit(global_max_b.value) == lit(global_min_b.value), lit(0.5)  # avoid divide by zero
    ).otherwise(
        (col("score") - lit(global_min_b.value)) / (lit(global_max_b.value) - lit(global_min_b.value))
    )
)

# 4. Clamp normalized scores to [0, 1]
recs_normalized = recs_normalized.withColumn(
    "normalized_score",
    when(col("normalized_score") < 0, 0.0)
    .when(col("normalized_score") > 1, 1.0)
    .otherwise(col("normalized_score"))
)

Global prediction min: 0.5189207196235657, max: 6.546607971191406


In [13]:
# 5. Convert item_ids into Long type to ensure consistency for later merging of the models
recs_fixed = recs_normalized.withColumn(
    "item_id",
    col("item_id").cast(LongType())
)

# 6. Reconstruct recommendations per user as list of [item_id, normalized_score]
user_recs_normalized = recs_fixed.groupBy("user_id").agg(
    collect_list(array("item_id", "normalized_score")).alias("recommendations_normalized")
)

# 7. Show results
user_recs_normalized.show(5, truncate=False)

+-------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

# Books
## Content-based Filtering

#### Short Summary for CBF
1. Loading of the data  
2. Normalizing the scores to be between 0 and 1 and filtering to keep only the scores higher or equal to 0.3
3. Model Training (LDA + K-means) 

### 1. Loading of the data

In [14]:
# Loading book_tagdl_filtered.csv
try: 
    book_tags = spark.read.option("header", "true").csv("./cross_domain_data/book_tagdl_filtered.csv")
except Exception as e:
    print(f"Error loading file: {e}")
    raise

In [15]:
book_tags.show(5)

+-------+--------+--------------------+
|    tag| item_id|               score|
+-------+--------+--------------------+
|samurai|16416771|2.401769200000000...|
|samurai|23756807|         1.886487E-4|
|samurai|24248331|          8.34465E-6|
|samurai|16646156|-1.11758710000000...|
|samurai|21856269|-1.04308130000000...|
+-------+--------+--------------------+
only showing top 5 rows



In [16]:
# Group tags and scores per item
df = book_tags.groupBy("item_id").agg(
    F.collect_list(F.struct("tag", "score")).alias("tag_vector")
)

In [17]:
df.show(2, truncate=False)

+--------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

### 2. Normalizing the scores (0-1) and then filtering all the scores higher or equal to 0.3

In [18]:
# 1. Explode tag_vector into flat structure
df_exploded = df.withColumn("tag_score", explode(col("tag_vector")))

# 2. Use dot notation to extract struct fields
df_flat = df_exploded.select(
    col("item_id"),
    col("tag_score.tag").alias("tag"),
    col("tag_score.score").cast("double").alias("score")
)

# 3. Compute global min and max
global_minmax = df_flat.agg(
    spark_min("score").alias("global_min"),
    spark_max("score").alias("global_max")
)

# 4. Cross join to add global_min and global_max
df_with_global = df_flat.crossJoin(global_minmax)

# 5. Normalize scores
df_normalized = df_with_global.withColumn(
    "normalized_score",
    when(
        col("global_max") == col("global_min"), lit(0.5)
    ).otherwise(
        (col("score") - col("global_min")) / (col("global_max") - col("global_min"))
    )
)

# 6. Filter
df_filtered = df_normalized.filter(col("normalized_score") >= 0.3)

# 7. Reconstruct tag_vector
book_tags_normalized = df_filtered.groupBy("item_id").agg(
    collect_list(array("tag", "normalized_score")).alias("tag_vector_filtered")
)

# Show result
book_tags_normalized.show(3, truncate=False)

+--------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|item_id |tag_vector_filtered                                                                                                                                                                                                                        |
+--------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|13134603|[[courage, 0.35317743176727295], [writers, 0.44461084844236004], [revenge, 0.4604257885640023], [prejudice, 0.35242162579058983], [thought-provoking, 0.3071030436159008], [interesting, 0.49520252665176995], [dark, 0.34127400970016]]   |
|13322179|[[

### Transforming the data into a form compatible with the model

In [19]:
# 1. Build documents (book -> tags list)
tags_exploded = book_tags_normalized.withColumn(
    "tag", F.expr("transform(tag_vector_filtered, x -> x[0])")
).select(
    "item_id", "tag"
).withColumn("tag", F.explode("tag"))

docs = tags_exploded.groupBy("item_id").agg(
    F.collect_list("tag").alias("tags")
)

# 2. Vectorize tags -> term-count vectors
cv = CountVectorizer(inputCol="tags", outputCol="features", vocabSize=10000, minDF=1.0)
cv_model = cv.fit(docs)
docs_featurized = cv_model.transform(docs)

## 3. Model Training (LDA + K-means) - Content-based Filtering

In [20]:
# 1. Train LDA -> get topic distributions
k_topics = 30
lda = LDA(k=k_topics, maxIter=20, featuresCol="features", seed=42)
lda_model = lda.fit(docs_featurized)
books_with_topics = lda_model.transform(docs_featurized)

# 2. Cluster books in topic space with KMeans
k_clusters = 200  # increase for finer groups & better chance of getting enough recommendations
kmeans = KMeans(featuresCol="topicDistribution", predictionCol="cluster", k=k_clusters, seed=42)
kmeans_model = kmeans.fit(books_with_topics)
books_with_clusters = kmeans_model.transform(books_with_topics)

In [21]:
# For every cluster, get all item_ids in a list
cluster_books = books_with_clusters.groupBy("cluster").agg(
    F.collect_list("item_id").alias("items_in_cluster")
)

# Join each book with its cluster's book list
books_with_recs = books_with_clusters.join(
    cluster_books, on="cluster", how="inner"
)


def exclude_self_and_limit(recs, self_id, limit=15):
    if recs is None or self_id is None:
        return []
    # Normalize types to avoid mismatch
    self_id = int(self_id)
    result = [int(i) for i in recs if int(i) != self_id]
    return result[:limit]

exclude_self_and_limit_udf = F.udf(
    exclude_self_and_limit,
    ArrayType(LongType())
)

books_cbf_recommendations = books_with_recs.withColumn(
    "recommended_items",
    exclude_self_and_limit_udf(F.col("items_in_cluster"), F.col("item_id"))
).select("item_id", "recommended_items")

books_cbf_recommendations.show(5, truncate=False)

+--------+------------------------------------------------------------------------------------------------------------------------------------------------+
|item_id |recommended_items                                                                                                                               |
+--------+------------------------------------------------------------------------------------------------------------------------------------------------+
|43134689|[50705784, 48637040, 18390691, 42883994, 64823, 19154394, 135328, 16910905, 25441287, 1882596, 18455512, 50566357, 21964544, 45338966, 26115712]|
|50705784|[43134689, 48637040, 18390691, 42883994, 64823, 19154394, 135328, 16910905, 25441287, 1882596, 18455512, 50566357, 21964544, 45338966, 26115712]|
|48637040|[43134689, 50705784, 18390691, 42883994, 64823, 19154394, 135328, 16910905, 25441287, 1882596, 18455512, 50566357, 21964544, 45338966, 26115712]|
|18390691|[43134689, 50705784, 48637040, 42883994, 64823, 191543

# Combining 2 models - BOOKS

#### Description
We first boost collaborative filtering (CF) recommendations by increasing their scores if they match books similar in content to those the user previously liked (identified using content-based filtering, CBF). We then ensure each user receives exactly 15 recommendations: if a user has fewer than 15 boosted CF recs after filtering and capping, we fill the gap with fallback items taken from CBF recommendations of their liked books, excluding books the user has already seen. Then, we combine the boosted and fallback items, rank them by score, limit them to the top 15 per user, and aggregate them into a final list of recommendations. Finally, we transform item_ids into titles from metadata and save the final file to be later used for reddit enrichment

In [22]:
# --- BOOSTING PHASE ---

# 1. Identify positive items per user
user_likes = train_df.filter(F.col("rating") >= 4).groupBy("user_id").agg(
    F.collect_set("item_id").alias("liked_items")
)

# 2. Expand liked items to get content-similar books
liked_items_exploded = user_likes.withColumn("liked_item", F.explode("liked_items"))

# 3. Join liked items with CBF recs
cbf_hits = liked_items_exploded.join(
    books_cbf_recommendations,
    liked_items_exploded.liked_item == books_cbf_recommendations.item_id,
    how="inner"
).select(
    "user_id",
    F.explode("recommended_items").alias("cbf_hit")
)

# 4. Build user -> set of all cbf hits
user_cbf_hits = cbf_hits.groupBy("user_id").agg(
    F.collect_set("cbf_hit").alias("cbf_hits")
)

# 5. Explode CF recommendations: user_id, cf_item, cf_score
cf_exploded = user_recs_normalized.withColumn(
    "rec", F.explode("recommendations_normalized")
).select(
    "user_id",
    F.col("rec").getItem(0).alias("cf_item"),
    F.col("rec").getItem(1).alias("cf_score")
)

# 6. Join exploded CF recs with user's cbf_hits
cf_with_hits = cf_exploded.join(
    user_cbf_hits, on="user_id", how="left"
)

# 7. Boost CF score if cf_item is in cbf_hits
cf_with_boost = cf_with_hits.withColumn(
    "boosted_score",
    F.when(
        (F.col("cbf_hits").isNotNull()) & (F.expr("array_contains(cbf_hits, cf_item)")),
        F.col("cf_score") + F.lit(0.1)
    ).otherwise(F.col("cf_score"))
).withColumn(
    "boosted_score",
    F.when(F.col("boosted_score") > 1.0, 1.0).otherwise(F.col("boosted_score"))
)

# 8. Re-rank boosted recs & cap at 15
window_rank = Window.partitionBy("user_id").orderBy(F.desc("boosted_score"))
cf_ranked = cf_with_boost.withColumn(
    "rank", F.row_number().over(window_rank)
)

recs_capped = cf_ranked.filter(F.col("rank") <= 15).select(
    "user_id", "cf_item", "boosted_score"
)


# --- FALLBACK PHASE ---

# 9. Count how many recs each user actually got after capping
user_num_recs = recs_capped.groupBy("user_id").agg(
    F.count("*").alias("num_recs")
)

users_with_gaps = user_num_recs.filter(F.col("num_recs") < 15).withColumn(
    "num_fallbacks", F.lit(15) - F.col("num_recs")
)

# 10. Build fallback candidates: liked items -> cbf recommendations
fallback_candidates = liked_items_exploded.join(
    books_cbf_recommendations,
    liked_items_exploded.liked_item == books_cbf_recommendations.item_id,
    how="inner"
).select(
    "user_id",
    F.explode("recommended_items").alias("fallback_item")
).distinct()

# 11. Exclude fallback items already seen
fallback_clean = fallback_candidates.join(
    user_seen_items, on="user_id", how="left"
).filter(
    ~F.expr("array_contains(seen_items, fallback_item)")
).select(
    "user_id", "fallback_item"
)

# 12. Rank fallback items per user, keep needed number
fallback_needed = users_with_gaps.join(fallback_clean, on="user_id", how="inner")

window_fallback = Window.partitionBy("user_id").orderBy("fallback_item")

fallback_ranked = fallback_needed.withColumn(
    "rank", F.row_number().over(window_fallback)
).filter(
    F.col("rank") <= F.col("num_fallbacks")
).select(
    "user_id", F.col("fallback_item").alias("cf_item")
).withColumn(
    "boosted_score", F.lit(0.0)  # fallback items have dummy low score
)


# --- FINAL AGGREGATION ---

# 13. Combine boosted recommendations with fallback
final_recs_all = recs_capped.unionByName(fallback_ranked)

# 14. Final re-rank & cap at 15 to ensure correct number per user
window_final = Window.partitionBy("user_id").orderBy(F.desc("boosted_score"))

final_ranked = final_recs_all.withColumn(
    "rank", F.row_number().over(window_final)
).filter(
    F.col("rank") <= 15
)

# 15. Aggregate back into list per user: [cf_item, boosted_score]
final_user_recs = final_ranked.groupBy("user_id").agg(
    F.collect_list(F.array("cf_item", "boosted_score")).alias("hybrid_recommendations")
)

# 16. Show examples
final_user_recs.show(5, truncate=False)

+-------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|user_id|hybrid_recommendations                                                                                                                                                                                                                                                                                                                                                                                                                                                             

#### Checking if every user has 15 recommendations

In [23]:
# 1) Compute number of recommendations per user
recs_with_counts = final_user_recs.withColumn(
    "num_recs", size(col("hybrid_recommendations"))
)

# 2) Filter users with fewer than 15 recommendations
users_with_few_recs = recs_with_counts.filter(col("num_recs") < 15)

# 3) Display some examples
users_with_few_recs.show(10, truncate=False)

# 4) Optionally: count how many users have too few
num_users_with_few = users_with_few_recs.count()
print(f"Number of users with fewer than 15 recommendations: {num_users_with_few}")


+-------+----------------------+--------+
|user_id|hybrid_recommendations|num_recs|
+-------+----------------------+--------+
+-------+----------------------+--------+

Number of users with fewer than 15 recommendations: 0


### Converting item_ids into titles

In [24]:
# Loading book_metadata_old.json
try: 
    book_metadata = spark.read.json("./within_domain_data/books_metadata_old.json")
except Exception as e:
    print(f"Error loading file: {e}")
    raise

In [25]:
book_metadata = book_metadata.select(
    col("item_id").cast("long"),  # cast item_id to match recs type
    col("title")
)
book_metadata.show(3)

+--------+--------------------+
| item_id|               title|
+--------+--------------------+
|16827462|The Fault in Our ...|
| 2792775|The Hunger Games ...|
| 8812783|Mockingjay (The H...|
+--------+--------------------+
only showing top 3 rows



In [26]:
# 1. Explode recommendations so we can join on item_id
recs_exploded = final_user_recs.withColumn(
    "rec", explode("hybrid_recommendations")
).select(
    "user_id",
    col("rec").getItem(0).alias("item_id"),
    col("rec").getItem(1).alias("score")
)

# 2. Join exploded recs with metadata to get titles
recs_with_titles = recs_exploded.join(
    book_metadata,
    on="item_id",
    how="left"
)

# 3. Replace item_id with title in recommendations
recs_with_titles = recs_with_titles.withColumn(
    "rec_with_title",
    array("title", "score")
)

# 4. Group back by user_id
final_user_recs_with_titles = recs_with_titles.groupBy("user_id").agg(
    collect_list("rec_with_title").alias("hybrid_recommendations_with_titles")
)

# 5. Show example result
final_user_recs_with_titles.show(3, truncate=False)

+-------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|user_id|hybrid_recommendations_with_titles       

### Saving the final file

In [27]:
# Convert nested array column to JSON string
final_user_recs_flat = final_user_recs.withColumn(
    "hybrid_recommendations_json",
    F.to_json("hybrid_recommendations")
).select("user_id", "hybrid_recommendations_json")

# Write the flattened DataFrame, not the original one!
final_user_recs_flat.coalesce(1).write.mode("overwrite").option("header", "true").csv("book_user_recs")

In [10]:
# Save the test data for the evaluation
test_df.coalesce(1).write.mode("overwrite").option("header", "true").csv("test_book_data")

# Movies
## Collaborative Filtering

#### Short Summary for CF
1. Loading of the data  
2. Splitting the data into train and test sets  
3. Model Training (ALS)  
4. Excluding books the user already rated  
5. Normalizing the scores to be between 0 and 1 

### 1. Loading of the data

In [11]:
# Loading movies_ratings_filtered_1980_plus.csv
try: 
    movie_ratings = spark.read.option("header", "true").csv("./within_domain_data/movies_ratings_filtered_1980_plus.csv")
except Exception as e:
    print(f"Error loading file: {e}")
    raise

movie_ratings.show(5)

+-------+------+-------+
|item_id|rating|user_id|
+-------+------+-------+
|      5|   3.0| 997206|
|     10|   4.0| 997206|
|     13|   4.0| 997206|
|     17|   5.0| 997206|
|     21|   4.0| 997206|
+-------+------+-------+
only showing top 5 rows



In [12]:
# Ensuring that the columns are all in the integer form
ratings_df2 = movie_ratings.select(
    col("item_id").cast("integer"),
    col("rating").cast("integer"),
    col("user_id").cast("integer"),
)

# Dropping any NAs
ratings_df2 = ratings_df2.dropna(subset=["user_id", "item_id"])

ratings_df2.show(10)

+-------+------+-------+
|item_id|rating|user_id|
+-------+------+-------+
|      5|     3| 997206|
|     10|     4| 997206|
|     13|     4| 997206|
|     17|     5| 997206|
|     21|     4| 997206|
|     28|     5| 997206|
|     31|     4| 997206|
|     39|     3| 997206|
|     40|     4| 997206|
|     45|     2| 997206|
+-------+------+-------+
only showing top 10 rows



## 2. Splitting the data into train and test sets

In [13]:
# 1. Add binary label for positive (rating >=4) or negative
ratings_labeled2 = ratings_df2.withColumn(
    "label", F.when(F.col("rating") >= 4, 1).otherwise(0)
)

# 2. Add a random number per rating (seeded for reproducibility)
ratings_labeled2 = ratings_labeled2.withColumn(
    "rand", F.rand(seed=42)
)

# 3. For positives: split 80% train, 20% test
pos_train2 = ratings_labeled2.filter((F.col("label") == 1) & (F.col("rand") <= 0.8))
pos_test2  = ratings_labeled2.filter((F.col("label") == 1) & (F.col("rand") > 0.8))

# 4. For negatives: same split
neg_train2 = ratings_labeled2.filter((F.col("label") == 0) & (F.col("rand") <= 0.8))
neg_test2  = ratings_labeled2.filter((F.col("label") == 0) & (F.col("rand") > 0.8))

# 5. Combine train and test splits separately
train_movie = pos_train2.union(neg_train2).drop("label", "rand")
test_movie = pos_test2.union(neg_test2).drop("label", "rand")

## 3. Model Training (ALS) - Collaborative Filtering

In [31]:
# ALS works with ratings DataFrame with columns: user, item, rating
als2 = ALS(
    userCol="user_id", 
    itemCol="item_id", 
    ratingCol="rating",
    rank=30,         # latent factors dimensionality
    maxIter=10, 
    regParam=0.1, 
    coldStartStrategy="drop",  # avoids NaNs in predictions
    implicitPrefs=False,       # we have explicit ratings
    nonnegative=True
)

als_model2 = als2.fit(train_movie)

In [32]:
user_recs2 = als_model2.recommendForAllUsers(30)  # top 30 recommendations per user
user_recs2.show(2, truncate=False)

+-------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|user_id|recommendations                                                                                                                                                                                                                                                                                                                                                                                                   

### 4. Excluding books the user already rated

In [33]:
# 1. Build mapping user -> set of items they already rated
user_seen_items2 = train_movie.groupBy("user_id").agg(
    collect_set("item_id").alias("seen_items")
)

# 2. Explode user recommendations (array of structs -> individual rows)
recs_exploded2 = user_recs2.withColumn(
    "rec", explode(col("recommendations"))
).select(
    "user_id",
    col("rec.item_id").alias("item_id"),
    col("rec.rating").cast("double").alias("score")
)

# 3. Join exploded recommendations with seen items per user
recs_with_seen2 = recs_exploded2.join(
    user_seen_items2, on="user_id", how="left"
)

# 4. Filter out recommendations the user has already rated
recs_filtered2 = recs_with_seen2.filter(
    ~array_contains(col("seen_items"), col("item_id"))
)

# 5. Reconstruct recommendations per user: list of [item_id, score]
user_recs_filtered2 = recs_filtered2.groupBy("user_id").agg(
    collect_list(array("item_id", "score")).alias("recommendations_filtered")
)

# 6. Show example results
user_recs_filtered2.show(5, truncate=False)

+-------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|user_id|recommendations_filtered                                                      

### 5. Normalizing the scores (0-1)

In [34]:
# 1. Compute global min and max predicted scores across all recommendations
global_minmax2 = recs_filtered2.agg(
    spark_min("score").alias("global_min"),
    spark_max("score").alias("global_max")
).collect()[0]

global_min2 = global_minmax2["global_min"]
global_max2 = global_minmax2["global_max"]

print(f"Global prediction min: {global_min2}, max: {global_max2}")

# 2. Broadcast min/max for distributed normalization
global_min_b2 = spark.sparkContext.broadcast(global_min2)
global_max_b2 = spark.sparkContext.broadcast(global_max2)

# 3. Normalize scores with min-max scaling
recs_normalized2 = recs_filtered2.withColumn(
    "normalized_score",
    when(
        lit(global_max_b2.value) == lit(global_min_b2.value), lit(0.5)  # avoid divide by zero
    ).otherwise(
        (col("score") - lit(global_min_b2.value)) / (lit(global_max_b2.value) - lit(global_min_b2.value))
    )
)

# 4. Clamp normalized scores to [0, 1]
recs_normalized2 = recs_normalized2.withColumn(
    "normalized_score",
    when(col("normalized_score") < 0, 0.0)
    .when(col("normalized_score") > 1, 1.0)
    .otherwise(col("normalized_score"))
)

# 5. Convert item_ids into Long type to ensure consistency for later merging of the models
recs_fixed2 = recs_normalized2.withColumn(
    "item_id",
    col("item_id").cast(LongType())
)

# 6. Reconstruct recommendations per user as list of [item_id, normalized_score]
user_recs_normalized2 = recs_fixed2.groupBy("user_id").agg(
    collect_list(array("item_id", "normalized_score")).alias("recommendations_normalized")
)

# 7. Show results
user_recs_normalized2.show(5, truncate=False)

Global prediction min: 1.5651874542236328, max: 8.37895679473877
+-------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

# Movies
## Content-based Filtering

#### Short Summary for CBF
1. Loading of the data  
2. Normalizing the scores to be between 0 and 1 and filtering to keep only the scores higher or equal to 0.65
3. Model Training (LDA + K-means)

### 1. Loading of the data

In [35]:
# Loading movie_tagdl_filtered_1980_plus.csv
try: 
    movie_tags = spark.read.option("header", "true").csv("./cross_domain_data/movie_tagdl_filtered_1980_plus.csv")
except Exception as e:
    print(f"Error loading file: {e}")
    raise

movie_tags.show(5)

+----+-------+------------+
| tag|item_id|       score|
+----+-------+------------+
|aids|     26| 0.005092442|
|aids|     29|0.0039109588|
|aids|    474|  0.00500077|
|aids|   1677|0.0018276572|
|aids|   1806|0.0041607916|
+----+-------+------------+
only showing top 5 rows



In [36]:
# Group tags and scores per item
df2 = movie_tags.groupBy("item_id").agg(
    F.collect_list(F.struct("tag", "score")).alias("tag_vector")
)

df2.show(5)

+-------+--------------------+
|item_id|          tag_vector|
+-------+--------------------+
|   2069|[[aids, 0.0065113...|
|   2088|[[aids, 0.0069856...|
|    296|[[aids, 0.0257088...|
|   2162|[[aids, 0.0037386...|
|   2294|[[aids, 0.0055308...|
+-------+--------------------+
only showing top 5 rows



### 2. Normalizing the scores (0-1) and then filtering all the scores higher or equal to 0.6

In [37]:
# 1. Explode tag_vector into flat structure
df_exploded2 = df2.withColumn("tag_score", explode(col("tag_vector")))

# 2. Use dot notation to extract struct fields
df_flat2 = df_exploded2.select(
    col("item_id"),
    col("tag_score.tag").alias("tag"),
    col("tag_score.score").cast("double").alias("score")
)

# 3. Compute global min and max
global_minmax2 = df_flat2.agg(
    spark_min("score").alias("global_min"),
    spark_max("score").alias("global_max")
)

# 4. Cross join to add global_min and global_max
df_with_global2 = df_flat2.crossJoin(global_minmax2)

# 5. Normalize scores
df_normalized2 = df_with_global2.withColumn(
    "normalized_score",
    when(
        col("global_max") == col("global_min"), lit(0.5)
    ).otherwise(
        (col("score") - col("global_min")) / (col("global_max") - col("global_min"))
    )
)

# 6. Filter
df_filtered2 = df_normalized2.filter(col("normalized_score") >= 0.6)

# 7. Reconstruct tag_vector
movie_tags_normalized = df_filtered2.groupBy("item_id").agg(
    collect_list(array("tag", "normalized_score")).alias("tag_vector_filtered")
)

# Show result
movie_tags_normalized.show(3, truncate=False)

+-------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

### Transforming the data into a form compatible with the model

In [38]:
# 1. Build documents (book -> tags list)
tags_exploded2 = movie_tags_normalized.withColumn(
    "tag", F.expr("transform(tag_vector_filtered, x -> x[0])")
).select(
    "item_id", "tag"
).withColumn("tag", F.explode("tag"))

docs2 = tags_exploded2.groupBy("item_id").agg(
    F.collect_list("tag").alias("tags")
)

# 2. Vectorize tags -> term-count vectors
cv2 = CountVectorizer(inputCol="tags", outputCol="features", vocabSize=10000, minDF=2.0)
cv_model2 = cv2.fit(docs2)
docs_featurized2 = cv_model2.transform(docs2)

## 3. Model Training (LDA + K-means) - Content-based Filtering

In [39]:
# 1. Train LDA -> get topic distributions
k_topics = 30
lda2 = LDA(k=k_topics, maxIter=20, featuresCol="features", seed=42)
lda_model2 = lda2.fit(docs_featurized2)
movies_with_topics = lda_model2.transform(docs_featurized2)

# 2. Cluster books in topic space with KMeans
k_clusters = 200  # increase for finer groups & better chance of getting enough recommendations
kmeans2 = KMeans(featuresCol="topicDistribution", predictionCol="cluster", k=k_clusters, seed=42)
kmeans_model2 = kmeans2.fit(movies_with_topics)
movies_with_clusters = kmeans_model2.transform(movies_with_topics)

In [40]:
# For every cluster, get all item_ids in a list
cluster_movies = movies_with_clusters.groupBy("cluster").agg(
    F.collect_list("item_id").alias("items_in_cluster")
)

# Join each book with its cluster's book list
movies_with_recs = movies_with_clusters.join(
    cluster_movies, on="cluster", how="inner"
)

def exclude_self_and_limit(recs, self_id, limit=15):
    if recs is None or self_id is None:
        return []
    # Normalize types to avoid mismatch
    self_id = int(self_id)
    result = [int(i) for i in recs if int(i) != self_id]
    return result[:limit]

exclude_self_and_limit_udf = F.udf(
    exclude_self_and_limit,
    ArrayType(LongType())
)

movies_cbf_recommendations = movies_with_recs.withColumn(
    "recommended_items",
    exclude_self_and_limit_udf(F.col("items_in_cluster"), F.col("item_id"))
).select("item_id", "recommended_items")

movies_cbf_recommendations.show(5, truncate=False)

+-------+------------------------------------------------------------------------+
|item_id|recommended_items                                                       |
+-------+------------------------------------------------------------------------+
|87660  |[44943, 34536]                                                          |
|44943  |[87660, 34536]                                                          |
|34536  |[87660, 44943]                                                          |
|89190  |[2826, 106489, 653, 3877, 2373, 4275, 100163, 55995, 77540, 5039, 2193] |
|2826   |[89190, 106489, 653, 3877, 2373, 4275, 100163, 55995, 77540, 5039, 2193]|
+-------+------------------------------------------------------------------------+
only showing top 5 rows



## Combining 2 models for MOVIES

#### Description
We first boost collaborative filtering (CF) recommendations by increasing their scores if they match movies similar in content to those the user previously liked (identified using content-based filtering, CBF). We then ensure each user receives exactly 15 recommendations: if a user has fewer than 15 boosted CF recs after filtering and capping, we fill the gap with fallback items taken from CBF recommendations of their liked movies, excluding movies the user has already seen. Then, we combine the boosted and fallback items, rank them by score, limit them to the top 15 per user, and aggregate them into a final list of recommendations. Finally, we transform item_ids into titles from metadata and save the final file to be later used for reddit enrichment

In [41]:
# --- BOOSTING PHASE ---

# 1. Identify positive items per user
user_likes2 = train_movie.filter(F.col("rating") >= 4).groupBy("user_id").agg(
    F.collect_set("item_id").alias("liked_items")
)

# 2. Expand liked items to get content-similar books
liked_items_exploded2 = user_likes2.withColumn("liked_item", F.explode("liked_items"))

# 3. Join liked items with CBF recs
cbf_hits2 = liked_items_exploded2.join(
    movies_cbf_recommendations,
    liked_items_exploded2.liked_item == movies_cbf_recommendations.item_id,
    how="inner"
).select(
    "user_id",
    F.explode("recommended_items").alias("cbf_hit")
)

# 4. Build user -> set of all cbf hits
user_cbf_hits2 = cbf_hits2.groupBy("user_id").agg(
    F.collect_set("cbf_hit").alias("cbf_hits")
)

# 5. Explode CF recommendations: user_id, cf_item, cf_score
cf_exploded2 = user_recs_normalized2.withColumn(
    "rec", F.explode("recommendations_normalized")
).select(
    "user_id",
    F.col("rec").getItem(0).alias("cf_item"),
    F.col("rec").getItem(1).alias("cf_score")
)

# 6. Join exploded CF recs with user's cbf_hits
cf_with_hits2 = cf_exploded2.join(
    user_cbf_hits2, on="user_id", how="left"
)

# 7. Boost CF score if cf_item is in cbf_hits
cf_with_boost2 = cf_with_hits2.withColumn(
    "boosted_score",
    F.when(
        (F.col("cbf_hits").isNotNull()) & (F.expr("array_contains(cbf_hits, cf_item)")),
        F.col("cf_score") + F.lit(0.1)
    ).otherwise(F.col("cf_score"))
).withColumn(
    "boosted_score",
    F.when(F.col("boosted_score") > 1.0, 1.0).otherwise(F.col("boosted_score"))
)

# 8. Re-rank boosted recs & cap at 15
window_rank2 = Window.partitionBy("user_id").orderBy(F.desc("boosted_score"))
cf_ranked2 = cf_with_boost2.withColumn(
    "rank", F.row_number().over(window_rank2)
)

recs_capped2 = cf_ranked2.filter(F.col("rank") <= 15).select(
    "user_id", "cf_item", "boosted_score"
)


# --- FALLBACK PHASE ---

# 9. Count how many recs each user actually got after capping
user_num_recs2 = recs_capped2.groupBy("user_id").agg(
    F.count("*").alias("num_recs")
)

users_with_gaps2 = user_num_recs2.filter(F.col("num_recs") < 15).withColumn(
    "num_fallbacks", F.lit(15) - F.col("num_recs")
)

# 10. Build fallback candidates: liked items -> cbf recommendations
fallback_candidates2 = liked_items_exploded2.join(
    movies_cbf_recommendations,
    liked_items_exploded2.liked_item == movies_cbf_recommendations.item_id,
    how="inner"
).select(
    "user_id",
    F.explode("recommended_items").alias("fallback_item")
).distinct()

# 11. Exclude fallback items already seen
fallback_clean2 = fallback_candidates2.join(
    user_seen_items2, on="user_id", how="left"
).filter(
    ~F.expr("array_contains(seen_items, fallback_item)")
).select(
    "user_id", "fallback_item"
)

# 12. Rank fallback items per user, keep needed number
fallback_needed2 = users_with_gaps2.join(fallback_clean2, on="user_id", how="inner")

window_fallback2 = Window.partitionBy("user_id").orderBy("fallback_item")

fallback_ranked2 = fallback_needed2.withColumn(
    "rank", F.row_number().over(window_fallback2)
).filter(
    F.col("rank") <= F.col("num_fallbacks")
).select(
    "user_id", F.col("fallback_item").alias("cf_item")
).withColumn(
    "boosted_score", F.lit(0.0)  # fallback items have dummy low score
)


# --- FINAL AGGREGATION ---

# 13. Combine boosted recommendations with fallback
final_recs_all2 = recs_capped2.unionByName(fallback_ranked2)

# 14. Final re-rank & cap at 15 to ensure correct number per user
window_final2 = Window.partitionBy("user_id").orderBy(F.desc("boosted_score"))

final_ranked2 = final_recs_all2.withColumn(
    "rank", F.row_number().over(window_final2)
).filter(
    F.col("rank") <= 15
)

# 15. Aggregate back into list per user: [cf_item, boosted_score]
final_user_recs2 = final_ranked2.groupBy("user_id").agg(
    F.collect_list(F.array("cf_item", "boosted_score")).alias("hybrid_recommendations")
)

# 16. Show examples
final_user_recs2.show(5, truncate=False)

+-------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|user_id|hybrid_recommendations                                                                                                                                                                                                                                                                                                                                                                                                                                                                |
+-------+-----------------------------

#### Checking if every user has 15 recommendations

In [42]:
# 1) Compute number of recommendations per user
recs_with_counts2 = final_user_recs2.withColumn(
    "num_recs", size(col("hybrid_recommendations"))
)

# 2) Filter users with fewer than 15 recommendations
users_with_few_recs2 = recs_with_counts2.filter(col("num_recs") < 15)

# 3) Display some examples
users_with_few_recs2.show(10, truncate=False)

# 4) Optionally: count how many users have too few
num_users_with_few2 = users_with_few_recs2.count()
print(f"Number of users with fewer than 15 recommendations: {num_users_with_few2}")


+-------+----------------------+--------+
|user_id|hybrid_recommendations|num_recs|
+-------+----------------------+--------+
+-------+----------------------+--------+

Number of users with fewer than 15 recommendations: 0


### Converting item_ids into titles

In [43]:
# Loading movies_metadata_filtered_1980_plus.csv
try: 
    movie_metadata = spark.read.option("header", "true").csv("./within_domain_data/movies_metadata_filtered_1980_plus.csv")
except Exception as e:
    print(f"Error loading file: {e}")
    raise

movie_metadata.show(5)

+---------+---------------+-------+-------+--------------------+--------------------+----+
|avgRating|     directedBy| imdbId|item_id|            starring|               title|year|
+---------+---------------+-------+-------+--------------------+--------------------+----+
|  3.89146|  John Lasseter|0114709|      1|Tim Allen, Tom Ha...|           Toy Story|1995|
|  3.26605|   Joe Johnston|0113497|      2|Jonathan Hyde, Br...|             Jumanji|1995|
|  3.17146|  Howard Deutch|0113228|      3|Jack Lemmon, Walt...|    Grumpier Old Men|1995|
|  2.86824|Forest Whitaker|0114885|      4|Angela Bassett, L...|   Waiting to Exhale|1995|
|   3.0762|  Charles Shyer|0113041|      5|Steve Martin, Mar...|Father of the Bri...|1995|
+---------+---------------+-------+-------+--------------------+--------------------+----+
only showing top 5 rows



In [44]:
movie_metadata = movie_metadata.select(
    col("item_id").cast("long"),  # cast item_id to match recs type
    col("title")
)
movie_metadata.show(3)

+-------+----------------+
|item_id|           title|
+-------+----------------+
|      1|       Toy Story|
|      2|         Jumanji|
|      3|Grumpier Old Men|
+-------+----------------+
only showing top 3 rows



In [45]:
# 1. Explode recommendations so we can join on item_id
recs_exploded2 = final_user_recs2.withColumn(
    "rec", explode("hybrid_recommendations")
).select(
    "user_id",
    col("rec").getItem(0).alias("item_id"),
    col("rec").getItem(1).alias("score")
)

# 2. Join exploded recs with metadata to get titles
recs_with_titles2 = recs_exploded2.join(
    movie_metadata,
    on="item_id",
    how="left"
)

# 3. Replace item_id with title in recommendations
recs_with_titles2 = recs_with_titles2.withColumn(
    "rec_with_title",
    array("title", "score")
)

# 4. Group back by user_id
final_user_recs_with_titles2 = recs_with_titles2.groupBy("user_id").agg(
    collect_list("rec_with_title").alias("hybrid_recommendations_with_titles")
)

# 5. Show example result
final_user_recs_with_titles2.show(3, truncate=False)

+-------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|user_id|hybrid_recommendations_with_titles                                                                                                                                                                                                                                                                                     

### Saving the final file

In [46]:
# Convert nested array column to JSON string
final_user_recs_flat2 = final_user_recs2.withColumn(
    "hybrid_recommendations_json",
    F.to_json("hybrid_recommendations")
).select("user_id", "hybrid_recommendations_json")

# Write the flattened DataFrame, not the original one!
final_user_recs_flat2.coalesce(1).write.mode("overwrite").option("header", "true").csv("movie_user_recs")

In [14]:
# Save the test data for the evaluation
test_movie.coalesce(1).write.mode("overwrite").option("header", "true").csv("test_movie_data")